# 03 — Plantes / K-Means (non supervisé)
**But :** regrouper les plantes d'extérieur adaptées au climat tempéré selon leurs caractéristiques horticoles **validées**.

L'ingestion et la revue qualité sont des étapes préalables. Aucun label artificiel. Si le catalogue est vide ou trop petit, le notebook s'arrête avec un message explicite. Les clusters sont exploratoires.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"database/scripts/plant_quality_report.py").exists())
load_dotenv(ROOT/".env",override=False)
load_dotenv(ROOT/".env.local",override=False)
DSN=os.environ["GARDNAIZE_ML_DATABASE_URL"].replace("postgresql+psycopg://","postgresql://",1)
def get_data():
    with psycopg.connect(DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("""SELECT plant_id,scientific_name,growth_form,exposure,soil_texture,
                water_requirement,hardiness_min_c,height_cm,spread_cm,evergreen,
                quality_status,ready_for_matching
                FROM horticulture.ml_plant_catalog
                WHERE quality_status='VALIDATED'
                  AND outdoor_confirmed IS TRUE
                  AND temperate_confirmed IS TRUE
                ORDER BY plant_id""")
            return pd.DataFrame(cur.fetchall(),columns=[d.name for d in cur.description])
df=get_data()
print("Plantes admissibles :",len(df))
display(df.head())



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
numeric=["hardiness_min_c","height_cm","spread_cm"]
categorical=["growth_form","exposure","soil_texture","water_requirement","evergreen"]
assert len(df)>=10, "Au moins 10 plantes VALIDATED d'extérieur/tempérées sont nécessaires."
assert df[numeric+categorical].notna().any(axis=1).sum()>=10, "Caractéristiques insuffisantes."
pre=ColumnTransformer([
    ("numeric",Pipeline([("impute",SimpleImputer(strategy="median",keep_empty_features=True)),("scale",StandardScaler())]),numeric),
    ("category",Pipeline([("impute",SimpleImputer(strategy="constant",fill_value="UNKNOWN")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),categorical)
],sparse_threshold=0)
X=pre.fit_transform(df)
scores={}
for k in range(2,min(8,len(df)-1)):
    yk=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(X)
    if len(set(yk))>1: scores[k]=silhouette_score(X,yk)
print("Silhouette :",scores)
assert scores,"Impossible de calculer des clusters distincts"
best=max(scores,key=scores.get)
model=KMeans(n_clusters=best,random_state=42,n_init=10).fit(X)
df["cluster"]=model.labels_
display(df.groupby("cluster").agg(n=("plant_id","size"),
    hardiness_mean=("hardiness_min_c","mean"),
    height_mean=("height_cm","mean"),
    spread_mean=("spread_cm","mean")).round(1))
xy=PCA(n_components=2).fit_transform(X)
plt.figure(figsize=(8,5))
plt.scatter(xy[:,0],xy[:,1],c=model.labels_,cmap="tab10")
plt.title("Plantes K-Means — PCA illustrative")
plt.show()

